In [73]:
import cv2
import numpy as np
import matplotlib.pyplot as plt 

def extract_and_rectify_document(image):
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)
    
    # Apply binary thresholding to focus on the edges of the paper
    _, thresholded = cv2.threshold(blurred, 100, 255, cv2.THRESH_BINARY)

    # cv2.imshow('thresholded', thresholded)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    # Perform morphological operations to clean up the image (optional)
    kernel = np.ones((5, 5), np.uint8)
    thresholded = cv2.morphologyEx(thresholded, cv2.MORPH_CLOSE, kernel)  # Close gaps in edges

    # Edge detection
    edges = cv2.Canny(thresholded, 75, 200)
    
    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # print(f"Number of contours detected: {len(contours)}")  # Debug: Print the number of contours
    
    if len(contours) == 0:
        #print("No contours found")
        return None, None

    # Find the largest contour (which should correspond to the paper)
    paper_contour = max(contours, key=cv2.contourArea)

    # image_with_contours = image.copy()
    # cv2.drawContours(image_with_contours, contours, -1, (0, 255, 0), 2)

    # cv2.imshow('image_with_contours', image_with_contours)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

    # image_with_paper_contour = image.copy()
    # cv2.drawContours(image_with_paper_contour, [paper_contour], -1, (0, 255, 0), 2)

    # cv2.imshow('image_with_paper_contours', image_with_paper_contour)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()
    
    
    
    # Approximate the contour to get corner points
    epsilon = 0.02 * cv2.arcLength(paper_contour, True)
    corners = cv2.approxPolyDP(paper_contour, epsilon, True)

    # print(f"Corners detected: {corners}")  # Debug: Print detected corners
    # print(len(corners))
    
    # Get perspective transform
    if len(corners) == 4:
        # Sort corners into consistent order
        corners = order_points(corners.reshape(4, 2))
        
        # Define output size (maintain aspect ratio)
        width = 800  # arbitrary width
        height = int(width * 1.414)  # A4 aspect ratio
        
        dst_points = np.array([
            [0, 0],
            [width - 1, 0],
            [width - 1, height - 1],
            [0, height - 1]
        ], dtype=np.float32)
        
        # Apply perspective transform
        transform_matrix = cv2.getPerspectiveTransform(corners, dst_points)
        warped = cv2.warpPerspective(image, transform_matrix, (width, height))
        
        return warped, len(corners)
    
    return None, None


In [74]:
import cv2
import numpy as np
import matplotlib.pyplot as plt 

def extract_and_rectify_document_draw_contours(image):
    # Convert to grayscale
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Apply Gaussian blur to reduce noise
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    brightened = cv2.convertScaleAbs(blurred, alpha=2.0, beta=0)

    # Apply histogram equalization to improve contrast
    equalized = cv2.equalizeHist(brightened)

    cv2.imshow('brightened', equalized)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
    # Apply binary thresholding to focus on the edges of the paper
    _, thresholded = cv2.threshold(equalized, 200, 255, cv2.THRESH_BINARY)

    cv2.imshow('thresholded', thresholded)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # Perform morphological operations to clean up the image (optional)
    kernel = np.ones((5, 5), np.uint8)
    thresholded = cv2.morphologyEx(thresholded, cv2.MORPH_CLOSE, kernel)  # Close gaps in edges

    # Edge detection
    edges = cv2.Canny(thresholded, 75, 200)
    
    # Find contours
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    print(f"Number of contours detected: {len(contours)}")  # Debug: Print the number of contours
    
    if len(contours) == 0:
        print("No contours found")
        return None

    # Find the largest contour (which should correspond to the paper)
    paper_contour = max(contours, key=cv2.contourArea)

    image_with_contours = image.copy()
    cv2.drawContours(image_with_contours, contours, -1, (0, 255, 0), 2)

    cv2.imshow('image_with_contours', image_with_contours)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    image_with_paper_contour = image.copy()
    cv2.drawContours(image_with_paper_contour, [paper_contour], -1, (0, 255, 0), 2)

    cv2.imshow('image_with_paper_contours', image_with_paper_contour)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [75]:

def extract_graph(warped_image):
    # Convert to grayscale if not already
    if len(warped_image.shape) == 3:
        gray = cv2.cvtColor(warped_image, cv2.COLOR_BGR2GRAY)
    else:
        gray = warped_image
    
    # Apply adaptive thresholding
    thresh = cv2.adaptiveThreshold(
        gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2
    )
    
    # Clean up noise
    kernel = np.ones((2,2), np.uint8)
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    
    return cleaned

def order_points(pts):
    # Initialize ordered points
    rect = np.zeros((4, 2), dtype=np.float32)
    
    # Top-left will have smallest sum
    # Bottom-right will have largest sum
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    
    # Top-right will have smallest difference
    # Bottom-left will have largest difference
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    
    return rect

In [76]:
import os

png_files = ['./data/train/'+f for f in os.listdir('./data/train/') if f.endswith('.png')]

In [77]:

# badimages = []

# for ind, imagepath in enumerate(png_files):
#     if ((ind % 100) == 0):
#         print(ind)

#     image = cv2.imread(imagepath)
#     if image is not None:
#         warped_image, num_corners = extract_and_rectify_document(image)
#         if num_corners == 4:
#             continue 
#         else: 
#             badimages.append(imagepath)
#     else:
#         print("HI")

In [78]:
# with open('bad_images.txt', 'w') as file:
#     file.writelines(badimages)

In [79]:
with open('bad_images.txt', 'r') as file:
    badimagesread = file.readlines()

In [80]:
badimages=badimagesread[0].split('./')

In [ ]:
ind = 200
image = cv2.imread(badimages[ind])
extract_and_rectify_document_draw_contours(image)